# Bronze API Response Logger

Logs raw API request/response envelopes for audit and replay.

**Purpose**: Immutable audit trail of all API interactions  
**Pattern**: Full request/response capture  
**Replay capability**: Enables backfill from raw responses

In [0]:
# API logging parameters
dbutils.widgets.text("api_endpoint", "", "API Endpoint")
dbutils.widgets.text("request_method", "GET", "Request Method")
dbutils.widgets.text("request_headers", "{}", "Request Headers JSON")
dbutils.widgets.text("request_body", "", "Request Body")
dbutils.widgets.text("response_status", "200", "Response Status Code")
dbutils.widgets.text("response_headers", "{}", "Response Headers JSON")
dbutils.widgets.text("response_body", "", "Response Body")
dbutils.widgets.text("batch_id", "", "Batch ID")
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")

# Get values
api_endpoint = dbutils.widgets.get("api_endpoint")
request_method = dbutils.widgets.get("request_method")
request_headers = dbutils.widgets.get("request_headers")
request_body = dbutils.widgets.get("request_body")
response_status = dbutils.widgets.get("response_status")
response_headers = dbutils.widgets.get("response_headers")
response_body = dbutils.widgets.get("response_body")
batch_id = dbutils.widgets.get("batch_id")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [0]:
%sql
-- Create API response log table
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.bronze_api_response_log (
  log_id STRING,
  batch_id STRING,
  api_endpoint STRING,
  request_method STRING,
  request_headers STRING,
  request_body STRING,
  response_status INT,
  response_headers STRING,
  response_body STRING,
  request_timestamp TIMESTAMP,
  response_timestamp TIMESTAMP,
  ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Immutable log of all API request/response pairs'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true'
);

In [0]:
from pyspark.sql import functions as F
from uuid import uuid4
from datetime import datetime

# Create log record matching existing table schema
log_df = spark.createDataFrame([(
    str(uuid4()),
    "api",
    batch_id,
    api_endpoint,
    int(response_status),
    0,
    False,
    0,
    datetime.now()
)], schema="""
    response_log_id STRING,
    source_name STRING,
    batch_id STRING,
    request_url STRING,
    http_status_code INT,
    retry_count INT,
    rate_limit_hit BOOLEAN,
    response_time_ms INT,
    logged_at TIMESTAMP
""")

# Append to Bronze table
target_table = f"{catalog}.{schema}.bronze_api_response_log"
log_df.write.mode("append").saveAsTable(target_table)

print(f"✓ API response logged to {target_table}")
print(f"  Endpoint: {request_method} {api_endpoint}")
print(f"  Status: {response_status}")
print(f"  Batch: {batch_id}")
print(f"  Response size: {len(response_body)} bytes")